# mmCIF, DSSR i podgląd motywu A-minor

Jak przejrzeć strukturę, adnotacje DSSR i sprawdzić ich jawne identyfikatory? Notebook korzysta z lokalnych plików przez bibliotekę `aminor_msa`; nie uruchamia pobierania ani nie rozpakowuje całego ZIP-a. Każdy notebook można uruchomić osobno przez **Restart Kernel → Run All**.

## Introduction

Zmień wybór w `settings.toml` albo w jawnie nazwanych zmiennych przy wywołaniu. Kontrolki działają w uruchomionym kernelu; po zmianie rodziny w kontrolce uruchom ponownie komórki, które odczytują `browser.selected_family`.

## Assumptions

- Środowisko: `uv sync --extra notebook`; w edytorze wybierz `.venv/bin/python`.
- Dane są już w `data/`. Jeśli ich brakuje, użyj z katalogu projektu `uv run python -m assets.scripts.download`; szczegóły w [acquisition.md](../../../docs/acquisition.md).
- To kontrola formatów i jawnych kluczy źródłowych. Nie powstaje mapowanie reszt struktury do kolumn MSA ani zbiór treningowy.
- Przykład wybiera Antoninę Olejnik, rodzinę Fluoride `RF01734` i strukturę `3vrs`. Inne rodziny mogą nie mieć struktur lub raportów DSSR.

## Notation

`pdb_id` wskazuje plik współrzędnych. `report_variant` jest literalnym sufiksem `.out…`; `model_number` jest numerem modelu w źródłowym mmCIF. Klucz reszty: `(model, auth_chain, auth_number, insertion, residue_name)`; label_chain/label_number pozostają osobnym systemem.

[Reader: odczyt, klucze i przygotowanie do treningu](../../../docs/reader.md).

In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd

from aminor_msa import DataStore
from aminor_msa.utils.config import read_toml
from aminor_msa.utils.paths import find_project_root
from aminor_msa.visualization import ViewStyle

ROOT = find_project_root()
NOTEBOOK_DIR = ROOT / "assets/analysis/base_functionality/26_10_06"
SETTINGS_PATH = NOTEBOOK_DIR / "settings.toml"
settings = read_toml(SETTINGS_PATH)
selection = settings["selection"]
limits = settings["display"]
style = ViewStyle(**{key: limits[key] for key in ("max_sequences", "max_columns", "font_size", "width", "height")})
store = DataStore(ROOT / "data")
display({"data_root": str(store.root), "selection": selection, "initial_state": store.load_state()})

## Select a catalogue link and a report

### Methodology

**Theoretical methodology.** Katalog Rfam.pdb wskazuje PDB, łańcuch i zakres. Raporty łączymy z plikiem współrzędnych po PDB w nazwie, a dokładne identyfikatory reszt weryfikujemy oddzielnie.

**Implementation methodology.** Najpierw wyświetlamy dostępne warianty i modele. Wczytujemy jeden plik `.cif.gz` oraz jeden członek ZIP-a. `.cif_document` to osobny, pełny odczyt kategorii mmCIF.

**Figure description.** Tabele pokazują wiersze katalogu dla wybranego PDB oraz dostępne sufiksy raportów; podsumowanie podaje liczbę modeli i treść oryginalnego polecenia DSSR.

In [ ]:
from aminor_msa.visualization import mappings_table, structures_table

family = store.family(selection["family"])
pdb_id = selection["pdb_id"]
report_variant = selection["report_variant"]
model_number = selection["model_number"]
source = store.structure(pdb_id)
mappings = mappings_table(family)
display(structures_table(family))
display(mappings.loc[mappings["pdb_id"] == source.pdb_id])
assert source.pdb_id in {row.pdb_id for row in family.mappings}, "PDB is not linked to the selected family."
display(pd.DataFrame([{"variant": handle.variant, "member": handle.member} for handle in source.reports]))
coordinates = source.coordinates
report_handle = source.report(report_variant)
report = report_handle.read()
display({"source_models": [model.num for model in coordinates.structure], "report_member": report_handle.member,
         "report_command": report.command, "report_complete": report.complete})
assert model_number in {model.num for model in coordinates.structure}

### Remarks

Jeśli wybrana rodzina nie ma struktur, wróć do katalogu i wybierz inną. Jeśli PDB nie ma raportu, nie traktuj go jako przykładu bez A-minor. Numer wariantu nie służy do wyboru modelu współrzędnych.

### Notes

## Inspect residues, atoms and full mmCIF

### Methodology

**Theoretical methodology.** mmCIF rozróżnia numerację autora i numerację label. Współrzędne obejmują także ligandy, wodę i altlocy. Brakująca reszta polimeru może istnieć w tabeli schematu, lecz nie mieć atomów.

**Implementation methodology.** `residues_table` ogranicza wiersze, zachowuje oba klucze. `atoms_table` pokazuje jeden natywny widok Gemmi. Pełny `cif_document` umożliwia dostęp do kategorii niewspółrzędnych.

**Figure description.** Tabela reszt wybranego łańcucha i modelu; tabela atomów pierwszej reszty. Osobna tabela podaje wiersze `_pdbx_poly_seq_scheme`, jeżeli ta kategoria istnieje.

In [ ]:
from aminor_msa.visualization import residues_table, atoms_table

chain = selection["chain"]  # Author chain, not label_asym_id.
residue_rows = limits["residue_rows"]
atom_rows = limits["atom_rows"]
residues = residues_table(coordinates, model_number=model_number, chain=chain, limit=residue_rows)
display(residues)
record = next(coordinates.iter_residues(model_number=model_number, chain=chain))
atoms = atoms_table(record)
display(atoms.head(atom_rows))
document = source.cif_document
block = document.sole_block()
scheme = block.find_mmcif_category("_pdbx_poly_seq_scheme.")
display({"entry": block.name, "title": block.find_value("_struct.title"), "polymer_scheme_rows": len(scheme)})
if len(scheme):
    display(pd.DataFrame([list(row) for row in scheme], columns=list(scheme.tags)).head(residue_rows))

### Remarks

`next(iter_residues(...))` wybiera pierwszy rekord w pliku. Możesz zmienić wybór, korzystając z dokładnego ID DSSR i `resolve_nucleotide`; przykład poniżej. Zmiany współrzędnych są możliwe przez natywny obiekt Gemmi; po zmianie identyfikatorów lub topologii wywołaj `clear_residue_index()`.

### Notes

## Inspect DSSR and audit identifiers

### Methodology

**Theoretical methodology.** `complete` oznacza obecność spójnej tabeli podsumowania nukleotydów, a nie pełne pokrycie biologiczne. Typy I/II/X oraz gwiazdki przy donorach pozostają zachowane.

**Implementation methodology.** `dssr_tables` zamienia obsługiwane sekcje na tabele. `residue_links_table` wyszukuje każdą resztę po dokładnym kluczu autora w jawnie wybranym modelu. Braki i niejednoznaczności zostają w wierszach diagnostycznych.

**Figure description.** Tabele nukleotydów, par, motywów, DBN i wyników dopasowania. Status matched potwierdza klucz, nie zgodność snapshotów ani atomów.

In [ ]:
from aminor_msa.visualization import dssr_tables, residue_links_table

row_limit = limits["residue_rows"]
raw_lines = limits["raw_report_lines"]
report_tables = dssr_tables(report)
for name, table in report_tables.items():
    display({"table": name, "rows": len(table)})
    display(table.head(row_limit))
links = residue_links_table(coordinates, report, model_number=model_number)
display(links)
display(links.groupby("status", dropna=False).size().rename("nucleotides").to_frame())
print("\n".join(report.raw_text.splitlines()[:raw_lines]))
display({"retained_source_sections": report.sections})
assert len(links) == len(report.nucleotides)
if not report.complete or links.empty or not links["status"].eq("matched").all():
    print("Review incomplete annotation or unresolved identifiers before downstream use.")

### Remarks

Nieobsługiwane sekcje są dostępne w `raw_text`, nie są wyrzucane. Wszystkie zgodne klucze nie dowodzą, że DSSR przetwarzał dokładnie ten sam snapshot RCSB. W tym etapie niczego nie przenosimy na kolumny MSA.

### Notes

## View coordinates and one motif

### Methodology

**Theoretical methodology.** Natywna struktura zostaje bez zmian. Podgląd pokazuje jeden model; motyw zaznacza donor i dwie reszty pary na podstawie dokładnych lookupów.

**Implementation methodology.** `view_structure` przekazuje lokalne współrzędne do py3Dmol. `view_aminor_motif` dodaje znaczniki na C1′, z pierwszym atomem jako fallback. Kolor pomarańczowy oznacza donor, niebieski partnerów pary.

**Figure description.** Obrotowy widok modelu i oznaczony triplet. Sfery służą nawigacji; ich rozmiar ani położenie nie są pomiarem wiązania wodorowego.

In [ ]:
from aminor_msa.visualization import view_structure, view_aminor_motif

chain = selection["chain"]
model_number = selection["model_number"]
width, height = style.width, style.height
viewer = view_structure(coordinates, model_number=model_number, chain=chain, width=width, height=height)
viewer.show()

In [ ]:
motif_index = selection["motif_index"]
model_number = selection["model_number"]
if report.aminor_motifs:
    motif = report.aminor_motifs[motif_index]
    donor = coordinates.resolve_nucleotide(motif.donor, model_number=model_number)
    display({"motif_type": motif.kind, "starred": motif.starred, "raw": motif.raw})
    display(atoms_table(donor).head(limits["atom_rows"]))
    motif_viewer = view_aminor_motif(coordinates, motif, model_number=model_number, style=style)
    motif_viewer.show()
else:
    print("This report contains no parsed A-minor motifs; select another report to inspect a triplet.")

### Remarks

3D wymaga JavaScript/WebGL oraz pobrania 3Dmol.js z CDN w przeglądarce. Współrzędne pochodzą z pliku lokalnego. Uruchomienie kernela sprawdza generowanie HTML; samo nie sprawdza renderowania WebGL.

### Notes

## Results / Summary

### LLM

W domyślnym 3vrs wariant 1 zawiera 52 nukleotydy i dwa motywy A-minor. Kontrola identyfikatorów znajduje 52 dokładne odpowiedniki w źródłowym modelu 1. Jest to weryfikacja kluczy; mapowanie do MSA i zgodność snapshotów pozostają osobnymi zadaniami.

### Person